In [ ]:
from vanna.openai.openai_chat import OpenAI_Chat
from vanna.chromadb.chromadb_vector import ChromaDB_VectorStore
import os
import pandas as pd
from sqlalchemy import create_engine, text


from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("VANNA_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = "gpt-4o-mini"


In [ ]:
# Connect to the database
db_name = "participants_mock2"
engine = create_engine(f"postgresql+psycopg2://{os.getenv('DB_USER')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{db_name}")
query = text("SELECT table_name, column_name, data_type FROM information_schema.columns WHERE table_schema = 'public';")
# Run the query and print results
with engine.connect() as conn:
    results = conn.execute(query)
    df_information_schema = pd.DataFrame(results.fetchall(), columns=results.keys())
df_information_schema.head()

,table_name,column_name,data_type
0,unstructured_data,Languages_embedding,USER-DEFINED
1,structured_data,Age,integer
2,structured_data,Total Years of Experience,double precision
3,unstructured_data,Work Experience/Company_embedding,USER-DEFINED
4,unstructured_data,Work Experience/Designation_embedding,USER-DEFINED


In [ ]:

class MyVanna(ChromaDB_VectorStore, OpenAI_Chat):
    def __init__(self, config=None):
        ChromaDB_VectorStore.__init__(self, config=config)
        OpenAI_Chat.__init__(self, config=config)

vn = MyVanna(config={"api_key": OPENAI_API_KEY, 'model': OPENAI_MODEL, "temperature": 0.0})

vn.connect_to_postgres(host='my-host', dbname='my-dbname', user='my-user', password='my-password', port='my-port')


plan = vn.get_training_plan_generic(df_information_schema)
plan

IndexError: list index out of range

In [ ]:


## ********************************************************************************************
## ********************************************************************************************
## ************* Simple Connecting SQLite to Vanna example ************************************
## ********************************************************************************************
## ********************************************************************************************


from sqlalchemy import create_engine, text

# Load your SQLite database file
engine = create_engine("sqlite:///data/participants.db")

# Query to get schema objects and their creation SQL
query = text("SELECT type, sql FROM sqlite_master WHERE sql IS NOT NULL")

# Run the query and print results
with engine.connect() as conn:
    result = conn.execute(query)
    for row in result:
        print(f"{row.type.upper()}: {row.sql}\n")

class MyVanna(ChromaDB_VectorStore, OpenAI_Chat):
    def __init__(self, config=None):
        ChromaDB_VectorStore.__init__(self, config=config)
        OpenAI_Chat.__init__(self, config=config)

vn = MyVanna(config={"api_key": OPENAI_API_KEY, 'model': OPENAI_MODEL, "temperature": 0.0})
vn.connect_to_sqlite("data/participants.db")
df_ddl = vn.run_sql("SELECT type, sql FROM sqlite_master WHERE sql is not null")
for ddl in df_ddl['sql'].to_list():
    #print(ddl)
    vn.train(ddl=ddl)

vn.train(question="Show all participants over 40", sql="SELECT * FROM participants_structured WHERE Age > 40")
training_data = vn.get_training_data()
print(training_data)
print(df_ddl['sql'][0])
vn.ask(question="Show all participants with over 3 years of experience")

In [2]:
import socket

def check_port(host, port, timeout=5):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            print(f"✅ Successfully connected to {host}:{port}")
            return True
    except (socket.timeout, ConnectionRefusedError, OSError) as e:
        print(f"❌ Cannot connect to {host}:{port} -> {e}")
        return False

# Test Supabase Pooler
host = "aws-0-ap-southeast-1.pooler.supabase.com"
port = 6543

check_port(host, port)

✅ Successfully connected to aws-0-ap-southeast-1.pooler.supabase.com:6543


True